In [78]:
import yfinance as yf
import pandas as pd
import numpy as np
import requests
import zipfile
import io
import plotly.graph_objects as go
import plotly_express as px

In [14]:
INSTRUMENT_MAP = {
    # S&P 500 — all variants map to same instrument
    "ADJUSTED INT RATE S&P 500 TOTL - CHICAGO MERCANTILE EXCHANGE" :    "SP500",
    "E-MINI S&P 500 - CHICAGO MERCANTILE EXCHANGE" :                    "SP500",
    "E-MINI S&P 500 STOCK INDEX - CHICAGO MERCANTILE EXCHANGE" :        "SP500",
    "MICRO E-MINI S&P 500 INDEX - CHICAGO MERCANTILE EXCHANGE" :        "SP500",
    "S&P 500 ANNUAL DIVIDEND INDEX - CHICAGO MERCANTILE EXCHANGE" :     "SP500",
    "S&P 500 Consolidated - CHICAGO MERCANTILE EXCHANGE" :              "SP500",
    "S&P 500 QUARTERLY DIVIDEND IND - CHICAGO MERCANTILE EXCHANGE" :    "SP500",
    "S&P 500 STOCK INDEX - CHICAGO MERCANTILE EXCHANGE" :               "SP500",
    "S&P 500 TOTAL RETURN INDEX - CHICAGO MERCANTILE EXCHANGE" :        "SP500",

    'BITCOIN - CHICAGO MERCANTILE EXCHANGE':                            'BTC',
    'BITCOIN CASH PERP STYLE - COINBASE DERIVATIVES, LLC':              'BTC',
    'MICRO BITCOIN - CHICAGO MERCANTILE EXCHANGE':                      'BTC',
    'NANO BITCOIN PERP STYLE - COINBASE DERIVATIVES, LLC':              'BTC',

    "ULTRA 10-YEAR U.S. T-NOTES - CHICAGO BOARD OF TRADE" :             "US10Y",
    "ULTRA UST 10Y - CHICAGO BOARD OF TRADE" :                          "US10Y",
    "UST 10Y NOTE - CHICAGO BOARD OF TRADE" :                           "US10Y",
    "10-YEAR U.S. TREASURY NOTES - CHICAGO BOARD OF TRADE" :            "US10Y",
    
    '3-MONTH SOFR - CHICAGO MERCANTILE EXCHANGE' :                      "3 Month US",
    'SOFR-3M - CHICAGO MERCANTILE EXCHANGE' :                           "3 Month US",


    'SOFR-1M - CHICAGO MERCANTILE EXCHANGE' :                           '1 Month US',
    '1-MONTH SOFR - CHICAGO MERCANTILE EXCHANGE' :                      '1 Month US',

    "E-MINI RUSSELL 2000 INDEX - CHICAGO MERCANTILE EXCHANGE":          "russel",
    "EMINI RUSSELL 1000 GROWTH - CHICAGO MERCANTILE EXCHANGE":          "russel",
    "EMINI RUSSELL 1000 VALUE INDEX - CHICAGO MERCANTILE EXCHANGE":     "russel",
    "MICRO E-MINI RUSSELL 2000 INDX - CHICAGO MERCANTILE EXCHANGE":     "russel",
    "RUSSELL 1000 VALUE INDEX MINI - ICE FUTURES U.S.":                 "russel",
    "RUSSELL 2000 ANNUAL DIVIDEND  - CHICAGO MERCANTILE EXCHANGE":      "russel",
    "RUSSELL 2000 MINI INDEX FUTURE - ICE FUTURES U.S.":                "russel",
    "RUSSELL E-MINI - CHICAGO MERCANTILE EXCHANGE":                     "russel",

    'BRITISH POUND - CHICAGO MERCANTILE EXCHANGE':                      'GBP',
    'BRITISH POUND STERLING - CHICAGO MERCANTILE EXCHANGE':             'GBP',
    'EURO FX/BRITISH POUND XRATE - CHICAGO MERCANTILE EXCHANGE':        'GBP',

    'JAPANESE YEN - CHICAGO MERCANTILE EXCHANGE':                       'YEN',

    'CANADIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE':                    "CAD",
    'SWISS FRANC - CHICAGO MERCANTILE EXCHANGE' :                       "CHF",

    "MICRO E-MINI NASDAQ-100 INDEX - CHICAGO MERCANTILE EXCHANGE" :     "NASDAQ",
    "NASDAQ MINI - CHICAGO MERCANTILE EXCHANGE" :                       "NASDAQ",
    "NASDAQ-100 Consolidated - CHICAGO MERCANTILE EXCHANGE" :           "NASDAQ",
    "NASDAQ-100 STOCK INDEX (MINI) - CHICAGO MERCANTILE EXCHANGE" :     "NASDAQ",
}

In [15]:
TICKER_MAP = {
    'SP500':        '^GSPC',
    'BTC':          'BTC-USD',
    'US10Y':        'IEF',
    '3 Month US' :  'SHV',
    '1 Month US' :  'SHV',
    'russel':       'IWM',
    'GBP':          'GBPUSD=X',
    'YEN':          'JPY=X',
    'CAD' :         'CAD=X',
    'CHF' :         'CHF=X',
    'NASDAQ':       'QQQ'
}

In [16]:
def download_cot_data(years, report_type='combined'):
    """
    Downloads raw CFTC Traders in Financial Futures (TFF) data.

    Args:
        years       : list of int — e.g. [2020, 2021, 2022]
        report_type : 'futures' or 'combined' (futures + options)

    Returns:
        pd.DataFrame — raw unmodified TFF data
    """
    prefix_map = {
        'futures':  'fut_fin_xls',
        'combined': 'com_fin_xls',
    }
    assert report_type in prefix_map, f"report_type must be one of {list(prefix_map.keys())}"
    prefix = prefix_map[report_type]

    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)',
        'Referer':    'https://www.cftc.gov/MarketReports/CommitmentsofTraders/index.htm',
    }

    dfs = []
    for year in years:
        url = f"https://www.cftc.gov/files/dea/history/{prefix}_{year}.zip"
        print(f"Fetching {report_type} COT {year} from {url}...")
        try:
            response = requests.get(url, headers=headers, timeout=30)
            response.raise_for_status()
            zf  = zipfile.ZipFile(io.BytesIO(response.content))
            xls = zf.open(zf.namelist()[0]).read()
            dfs.append(pd.read_excel(io.BytesIO(xls), engine='xlrd'))
            print(f"  ✓ {year} fetched successfully")
        except requests.HTTPError as e:
            print(f"  ✗ HTTP error for {year}: {e}")
        except zipfile.BadZipFile:
            print(f"  ✗ Bad zip file for {year}")
        except Exception as e:
            print(f"  ✗ Unexpected error for {year}: {e}")

    if not dfs:
        raise RuntimeError("No data fetched — check years and network connection")

    return pd.concat(dfs, ignore_index=True)

In [108]:
def wavg(group, conc_cols):
        weights = group['Open_Interest_All']
        total = weights.sum()
        if total == 0:
            return pd.Series({c: 0.0 for c in conc_cols})
        return pd.Series({c: (group[c] * weights).sum() / total for c in conc_cols})

In [106]:
def clean_financial_cot_data(df, instrument_map):
    ''' Dealer — financial institutions/dealers (often on the other side of client trades)
        Asset_Mgr — institutional investors like pension funds, mutual funds
        Lev_Money — leveraged money (hedge funds, CTAs)
        Other_Rept — other reportable traders
        NonRept — non-reportable (small traders below reporting thresholds)
        Traders - it's counting number of traders, knowing 47 dealers are long tells you less than knowing their total position size
        Concentration Report - Shows what % of open interest the top 4 and top 8 traders control'''
    
    # take only the columns we need
    df = df[[
        'Market_and_Exchange_Names', 
        'Report_Date_as_MM_DD_YYYY', 
        'Open_Interest_All', 

        # Dealer
        'Dealer_Positions_Long_All', 'Dealer_Positions_Short_All', 'Dealer_Positions_Spread_All',
        # Asset Manager
        'Asset_Mgr_Positions_Long_All', 'Asset_Mgr_Positions_Short_All', 'Asset_Mgr_Positions_Spread_All',
        # Hedge Fund / Leveraged
        'Lev_Money_Positions_Long_All', 'Lev_Money_Positions_Short_All', 'Lev_Money_Positions_Spread_All',
        # Other Reportable
        'Other_Rept_Positions_Long_All', 'Other_Rept_Positions_Short_All', 'Other_Rept_Positions_Spread_All',
        # Non-Reportable
        'NonRept_Positions_Long_All', 'NonRept_Positions_Short_All',

        # Concentration — top 4 and top 8 traders
        'Conc_Gross_LE_4_TDR_Long_All', 'Conc_Gross_LE_4_TDR_Short_All',
        'Conc_Gross_LE_8_TDR_Long_All', 'Conc_Gross_LE_8_TDR_Short_All',
    ]]


    df['Report_Date_as_MM_DD_YYYY'] = pd.to_datetime(df['Report_Date_as_MM_DD_YYYY'])
    df = df.sort_values('Report_Date_as_MM_DD_YYYY')

    df['Instrument'] = df['Market_and_Exchange_Names'].map(instrument_map)
    df = df[df['Instrument'].notna()].copy()

    conc_cols = [
        'Conc_Gross_LE_4_TDR_Long_All', 'Conc_Gross_LE_4_TDR_Short_All',
        'Conc_Gross_LE_8_TDR_Long_All', 'Conc_Gross_LE_8_TDR_Short_All',
    ]

    cols_to_sum = df.columns.values.tolist()
    cols_to_sum.remove('Market_and_Exchange_Names')
    cols_to_sum.remove('Report_Date_as_MM_DD_YYYY')
    cols_to_sum.remove('Instrument')
    for c in conc_cols:
        cols_to_sum.remove(c)

    # Sum position columns
    df_summed = (
        df.groupby(['Instrument', 'Report_Date_as_MM_DD_YYYY'])[cols_to_sum]
        .sum()
        .reset_index()
    )

    # Weighted average concentration by OI
    

    df_conc = (
        df.groupby(['Instrument', 'Report_Date_as_MM_DD_YYYY'])
        .apply(wavg, conc_cols)
        .reset_index()
    )

    df = df_summed.merge(df_conc, on=['Instrument', 'Report_Date_as_MM_DD_YYYY'])


    return df

In [117]:
def feature_engineering(cot_clean):
    cot_clean["Dealer Net"] = cot_clean['Dealer_Positions_Long_All'] - cot_clean['Dealer_Positions_Short_All']
    cot_clean["Asset Manager Net"] = cot_clean['Asset_Mgr_Positions_Long_All'] - cot_clean['Asset_Mgr_Positions_Short_All'] 
    cot_clean["Levered Net"] = cot_clean['Lev_Money_Positions_Long_All'] - cot_clean['Lev_Money_Positions_Short_All']
    cot_clean["Other Rept Positions Net"] = cot_clean['Other_Rept_Positions_Long_All'] - cot_clean['Other_Rept_Positions_Short_All']
    cot_clean["Non Rept Positions Net"] = cot_clean['NonRept_Positions_Long_All'] - cot_clean['NonRept_Positions_Short_All']
    
    # In feature_engineering:
    cot_clean["Dealer Net Pct OI"] = cot_clean["Dealer Net"] / cot_clean["Open_Interest_All"]
    cot_clean["AM Net Pct OI"] = cot_clean["Asset Manager Net"] / cot_clean["Open_Interest_All"]
    cot_clean["HF Net Pct OI"] = cot_clean["Levered Net"] / cot_clean["Open_Interest_All"]
    cot_clean["Other Rept Positions Net Pct OI"] = cot_clean['Other Rept Positions Net']/cot_clean["Open_Interest_All"]
    cot_clean["Non Rept Positions Net Pct OI"] = cot_clean['Non Rept Positions Net']/cot_clean["Open_Interest_All"]


    return cot_clean

In [118]:
def get_align_prices(cot_clean):
    instrument_list = cot_clean['Instrument'].unique().tolist()
    tickers_list = [TICKER_MAP[inst] for inst in instrument_list if inst in TICKER_MAP]
    tickers_list
    prices = yf.download(tickers_list,start = '2018-01-01' )['Close']
    prices = prices.ffill()
    prices = prices.bfill()
    prices
    all_dfs = []
    for instr in instrument_list:
        cot_specific_instrument = cot_clean[cot_clean['Instrument'] ==instr ]
        cot_specific_instrument = cot_specific_instrument.set_index('Report_Date_as_MM_DD_YYYY')
        ticker_name = TICKER_MAP[instr]
        price_specific_instrument = pd.DataFrame(prices[ticker_name])
        merged_df = cot_specific_instrument.merge(price_specific_instrument, left_index=True, right_index=True, how = 'left')
        merged_df = merged_df.rename(columns={ticker_name: 'Price'})
        all_dfs.append(merged_df)
    cot_data = pd.concat(all_dfs)
    cot_data = cot_data.reset_index()
    return cot_data
    

In [146]:
cot_raw = download_cot_data([2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026])
cot_clean = clean_financial_cot_data(cot_raw, INSTRUMENT_MAP)
cot_clean
cot_clean = feature_engineering(cot_clean)
cot_clean = get_align_prices(cot_clean)
cot_clean

Fetching combined COT 2018 from https://www.cftc.gov/files/dea/history/com_fin_xls_2018.zip...
  ✓ 2018 fetched successfully
Fetching combined COT 2019 from https://www.cftc.gov/files/dea/history/com_fin_xls_2019.zip...
  ✓ 2019 fetched successfully
Fetching combined COT 2020 from https://www.cftc.gov/files/dea/history/com_fin_xls_2020.zip...
  ✓ 2020 fetched successfully
Fetching combined COT 2021 from https://www.cftc.gov/files/dea/history/com_fin_xls_2021.zip...
  ✓ 2021 fetched successfully
Fetching combined COT 2022 from https://www.cftc.gov/files/dea/history/com_fin_xls_2022.zip...
  ✓ 2022 fetched successfully
Fetching combined COT 2023 from https://www.cftc.gov/files/dea/history/com_fin_xls_2023.zip...
  ✓ 2023 fetched successfully
Fetching combined COT 2024 from https://www.cftc.gov/files/dea/history/com_fin_xls_2024.zip...
  ✓ 2024 fetched successfully
Fetching combined COT 2025 from https://www.cftc.gov/files/dea/history/com_fin_xls_2025.zip...
  ✓ 2025 fetched successfully


/var/folders/hs/v5c0lrnj61z3tp745vdhh0m40000gn/T/ipykernel_65047/96025408.py:33: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/hs/v5c0lrnj61z3tp745vdhh0m40000gn/T/ipykernel_65047/96025408.py:63: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.

/var/folders/hs/v5c0lrnj61z3tp745vdhh0m40000gn/T/ipykernel_65047/1065241557.py:5: FutureWarning:

YF.download() has changed argument auto_adjust default to True

[*********************100%*******************

,Report_Date_as_MM_DD_YYYY,Instrument,Open_Interest_All,Dealer_Positions_Long_All,Dealer_Positions_Short_All,Dealer_Positions_Spread_All,Asset_Mgr_Positions_Long_All,Asset_Mgr_Positions_Short_All,Asset_Mgr_Positions_Spread_All,Lev_Money_Positions_Long_All,...,Asset Manager Net,Levered Net,Other Rept Positions Net,Non Rept Positions Net,Dealer Net Pct OI,AM Net Pct OI,HF Net Pct OI,Other Rept Positions Net Pct OI,Non Rept Positions Net Pct OI,Price
0,2018-07-03,1 Month US,5921,3312,609,373,0,0,0,1,...,0,-2793,0,90,0.456511,0.000000,-0.471711,0.000000,0.015200,90.289505
1,2018-07-10,1 Month US,6670,3481,683,800,0,0,0,35,...,0,-2658,0,-140,0.419490,0.000000,-0.398501,0.000000,-0.020990,90.305908
2,2018-07-17,1 Month US,8879,5557,2356,738,0,0,0,28,...,0,-3205,-25,29,0.360514,0.000000,-0.360964,-0.002816,0.003266,90.338631
3,2018-07-24,1 Month US,12655,7410,3562,1061,0,0,0,748,...,0,-4025,102,75,0.304070,0.000000,-0.318056,0.008060,0.005927,90.363205
4,2018-07-31,1 Month US,14365,9576,4733,794,0,0,0,242,...,0,-4944,0,101,0.337139,0.000000,-0.344170,0.000000,0.007031,90.395943
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4677,2026-03-10,russel,497841,116253,105045,11292,179122,134544,25906,91387,...,44578,-41132,-2483,-12170,0.022513,0.089543,-0.082621,-0.004988,-0.024446,252.910110
4678,2026-03-17,russel,586922,144519,109868,40897,178574,134742,28596,81044,...,43832,-58919,-4348,-15217,0.059039,0.074681,-0.100386,-0.007408,-0.025927,250.050003
4679,2026-03-24,russel,478081,122590,95247,8324,166950,135445,27855,88922,...,31505,-53774,746,-5819,0.057193,0.065899,-0.112479,0.001560,-0.012172,248.779999
4680,2026-03-31,russel,472964,140220,103142,7998,160678,148025,28789,74459,...,12653,-41710,-4092,-3929,0.078395,0.026753,-0.088189,-0.008652,-0.008307,248.000000


In [131]:
cot_clean.columns.values.tolist()

['Report_Date_as_MM_DD_YYYY',
 'Instrument',
 'Open_Interest_All',
 'Dealer_Positions_Long_All',
 'Dealer_Positions_Short_All',
 'Dealer_Positions_Spread_All',
 'Asset_Mgr_Positions_Long_All',
 'Asset_Mgr_Positions_Short_All',
 'Asset_Mgr_Positions_Spread_All',
 'Lev_Money_Positions_Long_All',
 'Lev_Money_Positions_Short_All',
 'Lev_Money_Positions_Spread_All',
 'Other_Rept_Positions_Long_All',
 'Other_Rept_Positions_Short_All',
 'Other_Rept_Positions_Spread_All',
 'NonRept_Positions_Long_All',
 'NonRept_Positions_Short_All',
 'Conc_Gross_LE_4_TDR_Long_All',
 'Conc_Gross_LE_4_TDR_Short_All',
 'Conc_Gross_LE_8_TDR_Long_All',
 'Conc_Gross_LE_8_TDR_Short_All',
 'Dealer Net',
 'Asset Manager Net',
 'Levered Net',
 'Other Rept Positions Net',
 'Non Rept Positions Net',
 'Dealer Net Pct OI',
 'AM Net Pct OI',
 'HF Net Pct OI',
 'Other Rept Positions Net Pct OI',
 'Non Rept Positions Net Pct OI',
 'Price']

In [121]:
cot_raw.columns.values.tolist()

['Market_and_Exchange_Names',
 'As_of_Date_In_Form_YYMMDD',
 'Report_Date_as_MM_DD_YYYY',
 'CFTC_Contract_Market_Code',
 'CFTC_Market_Code',
 'CFTC_Region_Code',
 'CFTC_Commodity_Code',
 'Open_Interest_All',
 'Dealer_Positions_Long_All',
 'Dealer_Positions_Short_All',
 'Dealer_Positions_Spread_All',
 'Asset_Mgr_Positions_Long_All',
 'Asset_Mgr_Positions_Short_All',
 'Asset_Mgr_Positions_Spread_All',
 'Lev_Money_Positions_Long_All',
 'Lev_Money_Positions_Short_All',
 'Lev_Money_Positions_Spread_All',
 'Other_Rept_Positions_Long_All',
 'Other_Rept_Positions_Short_All',
 'Other_Rept_Positions_Spread_All',
 'Tot_Rept_Positions_Long_All',
 'Tot_Rept_Positions_Short_All',
 'NonRept_Positions_Long_All',
 'NonRept_Positions_Short_All',
 'Change_in_Open_Interest_All',
 'Change_in_Dealer_Long_All',
 'Change_in_Dealer_Short_All',
 'Change_in_Dealer_Spread_All',
 'Change_in_Asset_Mgr_Long_All',
 'Change_in_Asset_Mgr_Short_All',
 'Change_in_Asset_Mgr_Spread_All',
 'Change_in_Lev_Money_Long_All',
 '

In [ ]:
def stretch_overview(cot_clean, lookback_weeks=104):
    overview = []
    for inst in cot_clean['Instrument'].unique():
        df = cot_clean[cot_clean['Instrument'] == inst].copy()
        window = df.tail(lookback_weeks)
        

        latest = window.iloc[-1]
        dlr_pctl = (window['Dealer Net Pct OI'] < latest['Dealer Net Pct OI']).mean() * 100  
        am_pctl = (window['AM Net Pct OI'] < latest['AM Net Pct OI']).mean() * 100
        hf_pctl = (window['HF Net Pct OI'] < latest['HF Net Pct OI']).mean() * 100

        """
        1 Week stuff(quite noisy) ______________________
        """
        prev_1W = window.iloc[-2]
        
        dlr_chg_pct_oi_1W = float(latest['Dealer Net Pct OI'] - prev_1W['Dealer Net Pct OI'])
        am_chg_pct_oi_1W = float(latest['AM Net Pct OI'] - prev_1W['AM Net Pct OI'])
        hf_chg_pct_oi_1W = float(latest['HF Net Pct OI'] - prev_1W['HF Net Pct OI'])

        oi_chg_1W = int(latest['Open_Interest_All'] - prev_1W['Open_Interest_All'])
        oi_chg_pct_1W = (latest['Open_Interest_All'] - prev_1W['Open_Interest_All']) / prev_1W['Open_Interest_All']

        price_1W = float(latest['Price']) if pd.notna(latest.get('Price')) else None
        prev_price_1W = float(prev_1W['Price']) if pd.notna(prev_1W.get('Price')) else None
        price_chg_pct_1W =  ( ( (price_1W - prev_price_1W) / prev_price_1W)*100) if (price_1W and prev_price_1W) else None


        """
        1 Month stuff(Less noisy) ______________________
        """
        prev_1M = window.iloc[-5]
        

        dlr_chg_pct_oi_1M = float(latest['Dealer Net Pct OI'] - prev_1M['Dealer Net Pct OI'])
        am_chg_pct_oi_1M = float(latest['AM Net Pct OI'] - prev_1M['AM Net Pct OI'])
        hf_chg_pct_oi_1M = float(latest['HF Net Pct OI'] - prev_1M['HF Net Pct OI'])

        oi_chg_1M = int(latest['Open_Interest_All'] - prev_1M['Open_Interest_All'])
        oi_chg_pct_1M = (latest['Open_Interest_All'] - prev_1M['Open_Interest_All']) / prev_1M['Open_Interest_All']

        price_1M = float(latest['Price']) if pd.notna(latest.get('Price')) else None
        prev_price_1M = float(prev_1M['Price']) if pd.notna(prev_1M.get('Price')) else None
        price_chg_pct_1M =  ( ( (price_1M - prev_price_1M) / prev_price_1M)*100) if (price_1M and prev_price_1M) else None



        conc_cols = [
            'Conc_Gross_LE_4_TDR_Long_All', 'Conc_Gross_LE_4_TDR_Short_All',
            'Conc_Gross_LE_8_TDR_Long_All', 'Conc_Gross_LE_8_TDR_Short_All',
        ]

        conc = {}
        for col in conc_cols:
            if col in latest.index and pd.notna(latest[col]):
                conc[col] = float(latest[col])
                conc[f'{col}_chg_1W'] = round(float(latest[col] - prev_1W[col]), 2) if pd.notna(prev_1W[col]) else None
                conc[f'{col}_chg_1M'] = round(float(latest[col] - prev_1M[col]), 2) if pd.notna(prev_1M[col]) else None 

        overview.append({
                'instrument': inst,
                'ticker': TICKER_MAP.get(inst, ''),
                'latest_date': latest['Report_Date_as_MM_DD_YYYY'].strftime('%Y-%m-%d') if hasattr(latest['Report_Date_as_MM_DD_YYYY'], 'strftime') else str(latest['Report_Date_as_MM_DD_YYYY'])[:10],
                'oi': int(latest['Open_Interest_All']),

                # Price
                'price': price_1W,
                'price_chg_pct_1W': float(price_chg_pct_1W) if price_chg_pct_1W is not None else None,
                'price_chg_pct_1M': float(price_chg_pct_1M) if price_chg_pct_1M is not None else None,

                # Dealer — positioning
                'dealer_net': int(latest['Dealer Net']),
                'dealer_net_pct_oi': float(latest['Dealer Net Pct OI']),
                'dealer_pctl': float(dlr_pctl),
                'dealer_chg_pct_oi_1W': dlr_chg_pct_oi_1W,
                'dealer_chg_pct_oi_1M': dlr_chg_pct_oi_1M,

                # Asset Manager — positioning
                'am_net': int(latest['Asset Manager Net']),
                'am_net_pct_oi': float(latest['AM Net Pct OI']),
                'am_pctl': float(am_pctl),
                'am_chg_pct_oi_1W': am_chg_pct_oi_1W,
                'am_chg_pct_oi_1M': am_chg_pct_oi_1M,

                # Hedge Fund — positioning
                'hf_net': int(latest['Levered Net']),
                'hf_net_pct_oi': float(latest['HF Net Pct OI']),
                'hf_pctl': float(hf_pctl),
                'hf_chg_pct_oi_1W': hf_chg_pct_oi_1W,
                'hf_chg_pct_oi_1M': hf_chg_pct_oi_1M,

                # OI
                'oi_chg_1W': oi_chg_1W,
                'oi_chg_pct_1W': float(oi_chg_pct_1W),
                'oi_chg_1M': oi_chg_1M,
                'oi_chg_pct_1M': float(oi_chg_pct_1M),

                # Concentration
                **conc,
            })

    return overview  

In [183]:
overview_json = stretch_overview(cot_clean, 104)
overview_json

[{'instrument': '1 Month US',
  'ticker': 'SHV',
  'latest_date': '2026-04-07',
  'oi': 1302700,
  'price': 110.13999938964844,
  'price_chg_pct_1W': 0.06723011310172043,
  'price_chg_pct_1M': 0.257960101189475,
  'dealer_net': 130404,
  'dealer_net_pct_oi': 0.10010286328394873,
  'dealer_pctl': 42.30769230769231,
  'dealer_chg_pct_oi_1W': -0.012511641472531296,
  'dealer_chg_pct_oi_1M': -0.14408467028640193,
  'am_net': 95144,
  'am_net_pct_oi': 0.07303600214938205,
  'am_pctl': 98.07692307692307,
  'am_chg_pct_oi_1W': 0.014817221697606117,
  'am_chg_pct_oi_1M': 0.03937028428283036,
  'hf_net': -235635,
  'hf_net_pct_oi': -0.18088201427803793,
  'hf_pctl': 32.69230769230769,
  'hf_chg_pct_oi_1W': -0.004684049391310174,
  'hf_chg_pct_oi_1M': 0.09905323833542526,
  'oi_chg_1W': -342319,
  'oi_chg_pct_1W': -0.20809425301470683,
  'oi_chg_1M': -65695,
  'oi_chg_pct_1M': -0.048008798629050824,
  'Conc_Gross_LE_4_TDR_Long_All': 34.0,
  'Conc_Gross_LE_4_TDR_Long_All_chg_1W': 3.6,
  'Conc_Gro

a read from csv should be here

In [ ]:
df = cot_clean.copy()

features_list = df.columns.values.tolist()[2:]
features_list

['Open_Interest_All',
 'Dealer_Positions_Long_All',
 'Dealer_Positions_Short_All',
 'Dealer_Positions_Spread_All',
 'Asset_Mgr_Positions_Long_All',
 'Asset_Mgr_Positions_Short_All',
 'Asset_Mgr_Positions_Spread_All',
 'Lev_Money_Positions_Long_All',
 'Lev_Money_Positions_Short_All',
 'Lev_Money_Positions_Spread_All',
 'Other_Rept_Positions_Long_All',
 'Other_Rept_Positions_Short_All',
 'Other_Rept_Positions_Spread_All',
 'NonRept_Positions_Long_All',
 'NonRept_Positions_Short_All',
 'Conc_Gross_LE_4_TDR_Long_All',
 'Conc_Gross_LE_4_TDR_Short_All',
 'Conc_Gross_LE_8_TDR_Long_All',
 'Conc_Gross_LE_8_TDR_Short_All',
 'Dealer Net',
 'Asset Manager Net',
 'Levered Net',
 'Other Rept Positions Net',
 'Non Rept Positions Net',
 'Dealer Net Pct OI',
 'AM Net Pct OI',
 'HF Net Pct OI',
 'Other Rept Positions Net Pct OI',
 'Non Rept Positions Net Pct OI',
 'Price']

In [126]:
instrument_choice = 'BTC'
if instrument_choice not in df['Instrument'].unique():
    print('Invalid ticker')
df_filtered__Instrument = df[df['Instrument'] == instrument_choice]
df_filtered__Instrument = df_filtered__Instrument.set_index('Report_Date_as_MM_DD_YYYY')
df_filtered__Instrument

,Instrument,Open_Interest_All,Dealer_Positions_Long_All,Dealer_Positions_Short_All,Dealer_Positions_Spread_All,Asset_Mgr_Positions_Long_All,Asset_Mgr_Positions_Short_All,Asset_Mgr_Positions_Spread_All,Lev_Money_Positions_Long_All,Lev_Money_Positions_Short_All,...,Asset Manager Net,Levered Net,Other Rept Positions Net,Non Rept Positions Net,Dealer Net Pct OI,AM Net Pct OI,HF Net Pct OI,Other Rept Positions Net Pct OI,Non Rept Positions Net Pct OI,Price
Report_Date_as_MM_DD_YYYY,,,,,,,,,,,,,,,,,,,,,
2018-04-10,BTC,1702,0,0,0,122,0,0,902,1108,...,122,-206,-154,238,0.000000,0.071680,-0.121034,-0.090482,0.139835,6834.759766
2018-04-17,BTC,2029,0,0,0,122,0,0,1053,1313,...,122,-260,-202,340,0.000000,0.060128,-0.128142,-0.099556,0.167570,7902.089844
2018-04-24,BTC,2380,0,0,0,105,0,0,1194,1682,...,105,-488,-138,521,0.000000,0.044118,-0.205042,-0.057983,0.218908,9697.500000
2018-05-01,BTC,1991,0,0,0,0,0,0,1198,1348,...,0,-150,-106,256,0.000000,0.000000,-0.075339,-0.053240,0.128579,9119.009766
2018-05-08,BTC,2485,0,0,0,0,0,0,1284,1707,...,0,-423,23,400,0.000000,0.000000,-0.170221,0.009256,0.160966,9234.820312
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-03-10,BTC,246402,9149,5696,973,7218,2326,939,113538,73253,...,4892,40285,-59251,10621,0.014014,0.019854,0.163493,-0.240465,0.043104,69926.921875
2026-03-17,BTC,272735,9650,7643,2153,7294,2600,958,124973,102010,...,4694,22963,-39161,9496,0.007359,0.017211,0.084195,-0.143586,0.034818,73922.476562
2026-03-24,BTC,199093,8297,4960,3000,6574,2513,1137,33904,56205,...,4061,-22301,4997,9906,0.016761,0.020398,-0.112013,0.025099,0.049756,70517.859375


In [127]:
for feat in features_list:
    px.line(df_filtered__Instrument[feat], title = feat).show() 